# RA-KVC: rapid GPU experiment

Choose **Runtime > Change runtime type > GPU**, then run these cells in order. Upload the supplied project ZIP. No GitHub repository is needed. The default run evaluates 50 GSM8K questions with six methods on Qwen2.5-1.5B-Instruct, allowing up to 512 generated tokens. Real runtime depends on your GPU and answer lengths.

The ZIP already contains a smaller CPU pilot. This notebook produces a separate GPU study and rebuilds the report from actual new outputs.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one project ZIP"
base = Path("/content/ra_kvc_work").resolve()
base.mkdir(exist_ok=True)
with zipfile.ZipFile(archives[0]) as archive:
    for name in archive.namelist():
        target = (base/name).resolve()
        if base not in target.parents and target != base:
            raise ValueError("Unsafe archive path")
    archive.extractall(base)
root = base/"ra-kvc"
assert (root/"README.md").exists(), "Expected ra-kvc/ in ZIP"
os.chdir(root)
print("Project folder:", root)

In [ ]:
%pip -q install "transformers==4.46.3" "huggingface-hub==0.26.2" "tokenizers==0.20.3" "safetensors>=0.4.5" numpy matplotlib reportlab
%pip -q install -e . --no-deps
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime before running the model"
print(torch.__version__, torch.cuda.get_device_name(0))
print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
!python -m unittest discover -s tests -v

## Run the comparison

Start with 50 examples. Change `SAMPLES` before the first run if desired. Resuming uses the same output directory; changing settings requires a different directory. Calibration uses separate training questions. Answer generation is deterministic greedy decoding.

In [ ]:
import json, subprocess, sys
SAMPLES = 50
RUN = f"results/gpu_{SAMPLES}"
config = json.loads(Path("configs/fast_gpu.json").read_text())
config["samples"] = SAMPLES
config_path = Path("configs/colab_run.json")
config_path.write_text(json.dumps(config, indent=2))
subprocess.run([sys.executable, "scripts/run_benchmark.py", "--config", str(config_path), "--output", RUN], check=True)

## Analyze and build the report

This generates the CSV tables, PNG figures, an editable Markdown report, and a PDF. The diagnostic uses saved calibration trajectories. The report states whether the run is complete and does not manufacture missing results.

In [ ]:
subprocess.run([sys.executable, "scripts/analyze.py", "--run", RUN], check=True)
subprocess.run([sys.executable, "scripts/diagnose_signal.py", "--run", RUN], check=True)
subprocess.run([sys.executable, "scripts/profile_overhead.py", "--run", RUN], check=True)
subprocess.run([sys.executable, "scripts/validate_results.py", "--run", RUN, "--output", f"{RUN}/evidence_audit.json"], check=True)
subprocess.run([sys.executable, "scripts/build_report.py", "--run", RUN], check=True)
from IPython.display import display, Image
display(Image(filename=f"{RUN}/figures/accuracy.png"))
print(Path("report/RA_KVC_Report.md").read_text()[:3500])

## Optional HumanEval generation

The delivered pilot includes its own small code-evaluation run. A new GPU HumanEval run is optional. Generated code must be scored on the supplied Docker runner or after explicit review. Colab generally does not provide Docker, so this cell generates programs but does not execute them.

In [ ]:
RUN_HUMANEVAL = False
if RUN_HUMANEVAL:
    subprocess.run([sys.executable, "scripts/run_humaneval.py", "--run", RUN, "--output", "results/humaneval_gpu", "--limit", "40"], check=True)

## Download the updated project

The archive excludes downloaded model weights and datasets. Review the individual contribution section and update the editable presentation with your final results before recording the video.

In [ ]:
import zipfile
output_zip = Path("/content/RA_KVC_Updated_Project.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        relative = path.relative_to(root)
        if any(p in {"__pycache__", ".git", ".venv", "cache", "node_modules"} for p in relative.parts):
            continue
        if path.suffix in {".safetensors", ".bin", ".pyc"}:
            continue
        archive.write(path, Path("ra-kvc")/relative)
files.download(str(output_zip))
files.download(str(root/"report/RA_KVC_Report.pdf"))